# 📘 dbt (Data Build Tool) Concepts for Data Engineering
## Databricks Notebook — SQL-First Transformation Patterns

**What you'll learn:**
- What dbt is and why it dominates modern data stacks
- dbt project structure and conventions
- Models: staging → intermediate → mart (medallion in SQL)
- Materializations: view, table, incremental, ephemeral
- dbt-style testing (not_null, unique, accepted_values, relationships)
- Jinja macros → Python equivalent (reusable SQL generators)
- Incremental processing patterns
- dbt on Databricks integration

**Important:** dbt is a CLI tool that can't run inside Databricks Community Edition.
We teach the **patterns** dbt popularized and implement them using `spark.sql()` + Delta tables.
You'll understand dbt concepts AND be able to apply them anywhere.

**Prerequisites:** Notebooks 01-28 (techmart_dw database)

---

---
# 📗 Section 1: What is dbt and Why It Matters

## The Modern Data Stack

```
┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│   Sources   │───▶│  Ingestion  │───▶│  Transform  │───▶│  Analytics  │
│ (APIs, DBs) │    │ (Fivetran,  │    │   (dbt)     │    │ (Looker,    │
│             │    │  Airbyte)   │    │             │    │  Tableau)   │
└─────────────┘    └─────────────┘    └─────────────┘    └─────────────┘
                        EL                  T                  BI
```

## What is dbt?

dbt (data build tool) is a **SQL-first transformation framework** that:
- Turns SELECT statements into tables/views (no INSERT/CREATE needed)
- Manages dependencies between models automatically
- Provides built-in testing and documentation
- Tracks data lineage
- Enables software engineering practices for analytics

## Why dbt is Popular

| Feature | Without dbt | With dbt |
|---|---|---|
| SQL transforms | Manual CREATE TABLE AS | Just write SELECT |
| Dependencies | Manual ordering | Automatic via ref() |
| Testing | Custom scripts | Built-in assertions |
| Documentation | Separate wiki (stale) | Lives with code (always fresh) |
| Environments | Hardcoded schemas | Config-driven |
| Lineage | Unknown | Auto-generated DAG |

## ELT vs ETL

```
ETL (old): Extract → Transform (in Python/Spark) → Load to warehouse
ELT (new): Extract → Load to warehouse → Transform (in SQL via dbt)
```

dbt handles the **T** in ELT — transformations happen inside the data warehouse
using SQL, leveraging the warehouse's compute power.

In [ ]:
# dbt in a nutshell: a SELECT becomes a table
# In dbt, you write THIS (a .sql file):
dbt_model_example = """
-- models/marts/daily_sales.sql
{{ config(materialized='table') }}

SELECT
    DATE_TRUNC('day', o.order_date) AS sale_date,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(o.total_amount) AS total_revenue,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    SUM(o.total_amount) / COUNT(DISTINCT o.order_id) AS avg_order_value
FROM {{ ref('stg_orders') }} o
WHERE o.status IN ('completed', 'shipped')
GROUP BY 1
"""

# And dbt turns it into:
# CREATE TABLE analytics.daily_sales AS (SELECT ...)
# OR: CREATE VIEW analytics.daily_sales AS (SELECT ...)
# OR: MERGE INTO analytics.daily_sales ... (incremental)

print("📄 dbt Model (what you write):")
print(dbt_model_example)
print("🔄 What dbt does behind the scenes:")
print("   1. Parses ref('stg_orders') → resolves to actual table name")
print("   2. Determines execution order (stg_orders must run first)")
print("   3. Wraps your SELECT in CREATE TABLE/VIEW/MERGE")
print("   4. Executes against the warehouse")
print("   5. Runs tests you defined")
print("   6. Updates documentation & lineage graph")


In [ ]:
# Setup: verify our database is available
spark.sql("USE techmart_dw")
tables = [row.tableName for row in spark.sql("SHOW TABLES").collect()]
print(f"✅ techmart_dw ready with {len(tables)} tables")
print(f"   Tables: {', '.join(tables[:8])}...")

## dbt vs Traditional SQL Transformations

### The Problem dbt Solves

Without dbt, SQL transformations are:
- Scattered across notebooks, scripts, stored procedures
- No dependency management (run in wrong order = broken)
- No testing (bad data silently propagates)
- No documentation (what does this table mean?)
- No version control (who changed this query?)

With dbt:
- All transformations in one place (version-controlled)
- Dependencies auto-detected from `ref()` calls
- Built-in testing framework
- Auto-generated documentation
- Incremental processing built-in

### dbt's Core Innovation: ref()

```sql
-- Without dbt: hardcoded table names (fragile)
SELECT * FROM prod.silver.orders o
JOIN prod.silver.customers c ON o.customer_id = c.customer_id

-- With dbt: ref() resolves to correct environment
SELECT * FROM {{ ref('silver_orders') }} o
JOIN {{ ref('silver_customers') }} c ON o.customer_id = c.customer_id

-- In dev: resolves to dev.dbt_alice.silver_orders
-- In prod: resolves to prod.silver.silver_orders
-- Same code, different environments!
```

### dbt Materializations

| Materialization | What It Creates | When to Use |
|----------------|----------------|-------------|
| `view` | SQL view (no data stored) | Staging models, lightweight transforms |
| `table` | Physical table (full refresh) | Small-medium tables, final marts |
| `incremental` | Append/merge new rows only | Large tables, event data |
| `ephemeral` | CTE (no object created) | Intermediate logic, reusable CTEs |

```sql
-- Configure materialization in model file
{{ config(
    materialized='incremental',
    unique_key='order_id',
    incremental_strategy='merge'
) }}

SELECT
    order_id,
    customer_id,
    total_amount,
    status,
    order_date
FROM {{ ref('bronze_orders') }}

{% if is_incremental() %}
    -- Only process new records on incremental runs
    WHERE order_date > (SELECT MAX(order_date) FROM {{ this }})
{% endif %}
```

---
# 📗 Section 2: dbt Project Structure

## Standard dbt Project Layout

```
techmart_dbt/
├── dbt_project.yml              # Project configuration
├── profiles.yml                 # Connection settings (not in Git!)
├── packages.yml                 # External packages
├── models/
│   ├── staging/                 # 1:1 with sources, light cleaning
│   │   ├── _stg_models.yml     # Documentation + tests
│   │   ├── stg_orders.sql
│   │   ├── stg_customers.sql
│   │   └── stg_products.sql
│   ├── intermediate/            # Business logic, joins
│   │   ├── _int_models.yml
│   │   ├── int_order_items_enriched.sql
│   │   └── int_customer_orders.sql
│   └── marts/                   # Final business tables
│       ├── _mart_models.yml
│       ├── fct_daily_sales.sql
│       └── dim_customers.sql
├── tests/                       # Custom SQL tests
│   └── assert_positive_revenue.sql
├── macros/                      # Reusable SQL snippets
│   ├── generate_schema_name.sql
│   └── cents_to_dollars.sql
├── seeds/                       # Static CSV data
│   └── country_codes.csv
└── snapshots/                   # SCD Type 2 tracking
    └── snap_customers.sql
```

## Key Concepts

| Concept | Purpose | Example |
|---|---|---|
| **Model** | A SELECT that becomes a table/view | `stg_orders.sql` |
| **Source** | External table dbt reads from | Raw ingested data |
| **ref()** | Reference another model (creates dependency) | `{{ ref('stg_orders') }}` |
| **source()** | Reference a source table | `{{ source('raw', 'orders') }}` |
| **Materialization** | How the model is persisted | table, view, incremental |
| **Test** | Assertion about data quality | `unique`, `not_null` |
| **Macro** | Reusable SQL template | `{{ cents_to_dollars('amount') }}` |

In [ ]:
# Represent a dbt project structure in Python
dbt_project_config = {
    "name": "techmart_dbt",
    "version": "1.0.0",
    "config-version": 2,
    "profile": "techmart",
    "model-paths": ["models"],
    "test-paths": ["tests"],
    "macro-paths": ["macros"],
    "seed-paths": ["seeds"],
    "snapshot-paths": ["snapshots"],
    "models": {
        "techmart_dbt": {
            "staging": {
                "+materialized": "view",
                "+schema": "staging"
            },
            "intermediate": {
                "+materialized": "ephemeral"
            },
            "marts": {
                "+materialized": "table",
                "+schema": "marts"
            }
        }
    }
}

sources_config = {
    "version": 2,
    "sources": [
        {
            "name": "raw_techmart",
            "description": "Raw TechMart data loaded by ingestion pipeline",
            "database": "techmart_dw",
            "tables": [
                {"name": "orders", "description": "Raw order transactions"},
                {"name": "customers", "description": "Raw customer records"},
                {"name": "products", "description": "Product catalog"},
                {"name": "order_items", "description": "Line items per order"},
                {"name": "payments", "description": "Payment transactions"}
            ]
        }
    ]
}

import json
print("📄 dbt_project.yml:")
print(json.dumps(dbt_project_config, indent=2))
print("
📄 models/staging/_sources.yml:")
print(json.dumps(sources_config, indent=2))


In [ ]:
# ============================================
# 🎯 YOUR TURN: Design a dbt Project Structure
# ============================================
# Design a dbt project for a "FinanceCorps" company with:
# - Sources: transactions, accounts, exchange_rates, customers
# - Staging models: one per source (light cleaning)
# - Intermediate: int_transactions_enriched (join with exchange rates)
# - Marts: fct_daily_revenue, dim_accounts, dim_customers
#
# Create the project config as a Python dict showing:
# 1. The dbt_project.yml equivalent
# 2. The sources.yml equivalent
# 3. A list of all model files that would exist
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
finance_project = {
    "name": "financecorps_dbt",
    "version": "1.0.0",
    "models": {
        "financecorps_dbt": {
            "staging": {"+materialized": "view"},
            "intermediate": {"+materialized": "ephemeral"},
            "marts": {"+materialized": "table"}
        }
    }
}

finance_sources = {
    "sources": [{
        "name": "raw_finance",
        "tables": [
            {"name": "transactions", "description": "Raw financial transactions"},
            {"name": "accounts", "description": "Bank/trading accounts"},
            {"name": "exchange_rates", "description": "Daily FX rates"},
            {"name": "customers", "description": "Customer master data"}
        ]
    }]
}

model_files = {
    "staging": [
        "stg_transactions.sql",
        "stg_accounts.sql",
        "stg_exchange_rates.sql",
        "stg_customers.sql"
    ],
    "intermediate": [
        "int_transactions_enriched.sql"  # joins transactions + exchange_rates
    ],
    "marts": [
        "fct_daily_revenue.sql",
        "dim_accounts.sql",
        "dim_customers.sql"
    ]
}

print("📄 dbt_project.yml:")
print(json.dumps(finance_project, indent=2))
print("\n📄 sources.yml:")
print(json.dumps(finance_sources, indent=2))
print("\n📁 Model files:")
for layer, files in model_files.items():
    print(f"  models/{layer}/")
    for f in files:
        print(f"    └── {f}")


## dbt Models -- The Three Layers

### Staging Models (stg_*)

Staging models are 1:1 with source tables. They:
- Rename columns to consistent conventions
- Cast types properly
- Apply light cleaning (trim, lower)
- Add NO business logic

```sql
-- models/staging/stg_orders.sql
{{ config(materialized='view') }}

SELECT
    order_id,
    customer_id,
    CAST(total_amount AS DECIMAL(12,2)) AS order_total,
    LOWER(TRIM(status)) AS order_status,
    CAST(order_date AS DATE) AS ordered_at,
    _ingested_at AS loaded_at
FROM {{ source('raw', 'orders') }}
WHERE order_id IS NOT NULL
```

### Intermediate Models (int_*)

Intermediate models contain business logic:
- Joins between staging models
- Business calculations
- Enrichment

```sql
-- models/intermediate/int_order_items_enriched.sql
{{ config(materialized='ephemeral') }}

SELECT
    oi.order_item_id,
    oi.order_id,
    oi.product_id,
    p.product_name,
    p.category,
    oi.quantity,
    oi.unit_price,
    oi.net_amount,
    p.unit_cost * oi.quantity AS total_cost,
    oi.net_amount - (p.unit_cost * oi.quantity) AS profit
FROM {{ ref('stg_order_items') }} oi
LEFT JOIN {{ ref('stg_products') }} p ON oi.product_id = p.product_id
```

### Mart Models (fct_* and dim_*)

Mart models are the final output for business users:

```sql
-- models/marts/fct_daily_sales.sql
{{ config(materialized='table') }}

SELECT
    DATE_TRUNC('day', ordered_at) AS sale_date,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(order_total) AS total_revenue,
    AVG(order_total) AS avg_order_value
FROM {{ ref('stg_orders') }}
WHERE order_status IN ('completed', 'shipped')
GROUP BY 1
```

In [ ]:
# dbt model simulation -- implement the three-layer pattern
from pyspark.sql.functions import col, lower, trim, count, countDistinct, sum as spark_sum, avg, round as spark_round, date_trunc

print("dbt MODEL SIMULATION -- Three-Layer Pattern")
print("=" * 60)

# ─── STAGING MODELS (views) ───────────────────────────────────
print("\n1. STAGING MODELS (views -- light cleaning only)")

# stg_orders
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.stg_orders AS
SELECT
    order_id,
    customer_id,
    CAST(total_amount AS DECIMAL(12,2)) AS order_total,
    LOWER(TRIM(status)) AS order_status,
    CAST(order_date AS DATE) AS ordered_at,
    payment_method,
    order_source
FROM techmart_dw.orders
WHERE order_id IS NOT NULL
""")

# stg_customers
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.stg_customers AS
SELECT
    customer_id,
    LOWER(TRIM(email)) AS email,
    first_name,
    last_name,
    CONCAT(first_name, ' ', last_name) AS full_name,
    customer_segment,
    CAST(lifetime_value AS DECIMAL(12,2)) AS lifetime_value,
    CAST(registration_date AS DATE) AS signed_up_at
FROM techmart_dw.customers
WHERE customer_id IS NOT NULL
""")

print("   Created: stg_orders, stg_customers")
print(f"   stg_orders: {spark.table('techmart_dw.stg_orders').count():,} rows")
print(f"   stg_customers: {spark.table('techmart_dw.stg_customers').count():,} rows")

# ─── INTERMEDIATE MODELS (ephemeral = CTEs) ───────────────────
print("\n2. INTERMEDIATE MODELS (ephemeral -- business logic)")

spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.int_customer_orders AS
SELECT
    c.customer_id,
    c.full_name,
    c.email,
    c.customer_segment,
    c.lifetime_value,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(o.order_total) AS total_spent,
    AVG(o.order_total) AS avg_order_value,
    MAX(o.ordered_at) AS last_order_date
FROM techmart_dw.stg_customers c
LEFT JOIN techmart_dw.stg_orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.full_name, c.email, c.customer_segment, c.lifetime_value
""")

print("   Created: int_customer_orders")
print(f"   int_customer_orders: {spark.table('techmart_dw.int_customer_orders').count():,} rows")

# ─── MART MODELS (tables -- final output) ─────────────────────
print("\n3. MART MODELS (tables -- business-ready)")

spark.sql("DROP TABLE IF EXISTS techmart_dw.fct_daily_sales")
spark.sql("""
CREATE TABLE techmart_dw.fct_daily_sales AS
SELECT
    ordered_at AS sale_date,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(SUM(order_total), 2) AS total_revenue,
    ROUND(AVG(order_total), 2) AS avg_order_value
FROM techmart_dw.stg_orders
WHERE order_status IN ('completed', 'shipped')
GROUP BY ordered_at
ORDER BY ordered_at DESC
""")

print("   Created: fct_daily_sales")
print(f"   fct_daily_sales: {spark.table('techmart_dw.fct_daily_sales').count():,} rows")
print("\nSample fct_daily_sales:")
spark.table("techmart_dw.fct_daily_sales").show(5)


---
# 📗 Section 3: dbt Models → Equivalent SQL/Python

## The Three Layers

| Layer | dbt Convention | Purpose | Materialization |
|---|---|---|---|
| **Staging** | `stg_<source>` | 1:1 with source, rename/cast/clean | View |
| **Intermediate** | `int_<description>` | Business logic, joins, enrichment | Ephemeral/View |
| **Marts** | `fct_` or `dim_` | Final business tables | Table |

## Staging Models

Staging models are the **first transformation** on raw data:
- One model per source table
- Rename columns to consistent conventions
- Cast types properly
- Light cleaning (trim, lower)
- NO business logic here

```sql
-- models/staging/stg_orders.sql (dbt syntax)
{{ config(materialized='view') }}

SELECT
    order_id,
    customer_id,
    CAST(total_amount AS DECIMAL(12,2)) AS order_total,
    LOWER(TRIM(status)) AS order_status,
    CAST(order_date AS DATE) AS ordered_at,
    _loaded_at AS loaded_at
FROM {{ source('raw_techmart', 'orders') }}
WHERE order_id IS NOT NULL
```

In [ ]:
# Implement dbt staging models as SQL views in Databricks
# This is the EQUIVALENT of what dbt does behind the scenes

# --- stg_orders ---
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.stg_orders AS
SELECT
    order_id,
    customer_id,
    CAST(total_amount AS DECIMAL(12,2)) AS order_total,
    LOWER(TRIM(status)) AS order_status,
    CAST(order_date AS DATE) AS ordered_at,
    order_source,
    shipping_address
FROM techmart_dw.orders
WHERE order_id IS NOT NULL
""")

# --- stg_customers ---
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.stg_customers AS
SELECT
    customer_id,
    LOWER(TRIM(email)) AS email,
    first_name,
    last_name,
    CONCAT(first_name, ' ', last_name) AS full_name,
    customer_segment,
    CAST(lifetime_value AS DECIMAL(12,2)) AS lifetime_value,
    CAST(signup_date AS DATE) AS signed_up_at,
    is_active
FROM techmart_dw.customers
WHERE customer_id IS NOT NULL
""")

# --- stg_products ---
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.stg_products AS
SELECT
    product_id,
    TRIM(product_name) AS product_name,
    category,
    CAST(price AS DECIMAL(10,2)) AS unit_price,
    CAST(cost AS DECIMAL(10,2)) AS unit_cost,
    CAST(price AS DECIMAL(10,2)) - CAST(cost AS DECIMAL(10,2)) AS margin,
    supplier
FROM techmart_dw.products
WHERE product_id IS NOT NULL
""")

# --- stg_order_items ---
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.stg_order_items AS
SELECT
    order_item_id,
    order_id,
    product_id,
    quantity,
    CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
    CAST(discount AS DECIMAL(5,2)) AS discount_amount,
    CAST(unit_price * quantity AS DECIMAL(12,2)) AS gross_amount,
    CAST(unit_price * quantity - discount AS DECIMAL(12,2)) AS net_amount
FROM techmart_dw.order_items
WHERE order_item_id IS NOT NULL
""")

# --- stg_payments ---
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.stg_payments AS
SELECT
    payment_id,
    order_id,
    LOWER(payment_method) AS payment_method,
    CAST(amount AS DECIMAL(12,2)) AS payment_amount,
    LOWER(payment_status) AS payment_status,
    CAST(payment_date AS DATE) AS paid_at
FROM techmart_dw.payments
WHERE payment_id IS NOT NULL
""")

print("✅ Staging views created (dbt equivalent: materialized='view'):")
staging_views = ["stg_orders", "stg_customers", "stg_products", "stg_order_items", "stg_payments"]
for v in staging_views:
    count = spark.table(f"techmart_dw.{v}").count()
    print(f"   {v}: {count:,} rows")

## Intermediate Models

Intermediate models contain **business logic** — joins, enrichment, calculations.
In dbt, these are often `ephemeral` (not persisted, used as CTEs).

```sql
-- models/intermediate/int_order_items_enriched.sql (dbt syntax)
{{ config(materialized='ephemeral') }}

SELECT
    oi.order_item_id,
    oi.order_id,
    oi.product_id,
    p.product_name,
    p.category AS product_category,
    oi.quantity,
    oi.unit_price,
    oi.net_amount,
    p.unit_cost * oi.quantity AS total_cost,
    oi.net_amount - (p.unit_cost * oi.quantity) AS profit
FROM {{ ref('stg_order_items') }} oi
LEFT JOIN {{ ref('stg_products') }} p ON oi.product_id = p.product_id
```

In [ ]:
# Intermediate models — we'll use views (closest to ephemeral in notebooks)

# --- int_order_items_enriched ---
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.int_order_items_enriched AS
SELECT
    oi.order_item_id,
    oi.order_id,
    oi.product_id,
    p.product_name,
    p.category AS product_category,
    oi.quantity,
    oi.unit_price,
    oi.net_amount,
    p.unit_cost * oi.quantity AS total_cost,
    oi.net_amount - (p.unit_cost * oi.quantity) AS profit
FROM techmart_dw.stg_order_items oi
LEFT JOIN techmart_dw.stg_products p ON oi.product_id = p.product_id
""")

# --- int_customer_orders ---
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.int_customer_orders AS
SELECT
    c.customer_id,
    c.full_name,
    c.email,
    c.customer_segment,
    c.lifetime_value,
    c.signed_up_at,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(o.order_total) AS total_spent,
    AVG(o.order_total) AS avg_order_value,
    MIN(o.ordered_at) AS first_order_date,
    MAX(o.ordered_at) AS last_order_date,
    DATEDIFF(CURRENT_DATE(), MAX(o.ordered_at)) AS days_since_last_order
FROM techmart_dw.stg_customers c
LEFT JOIN techmart_dw.stg_orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.full_name, c.email, c.customer_segment, 
         c.lifetime_value, c.signed_up_at
""")

# --- int_daily_order_metrics ---
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.int_daily_order_metrics AS
SELECT
    o.ordered_at AS order_date,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    SUM(o.order_total) AS gross_revenue,
    SUM(CASE WHEN o.order_status = 'completed' THEN o.order_total ELSE 0 END) AS completed_revenue,
    AVG(o.order_total) AS avg_order_value
FROM techmart_dw.stg_orders o
GROUP BY o.ordered_at
""")

print("✅ Intermediate views created (dbt equivalent: materialized='ephemeral'):")
int_views = ["int_order_items_enriched", "int_customer_orders", "int_daily_order_metrics"]
for v in int_views:
    count = spark.table(f"techmart_dw.{v}").count()
    print(f"   {v}: {count:,} rows")

## Mart Models (Final Business Tables)

Mart models are the **final output** — what business users query.
- `fct_` prefix = fact tables (events, transactions)
- `dim_` prefix = dimension tables (entities, attributes)

```sql
-- models/marts/fct_daily_sales.sql (dbt syntax)
{{ config(materialized='table') }}

SELECT
    order_date,
    total_orders,
    unique_customers,
    gross_revenue,
    completed_revenue,
    avg_order_value,
    CURRENT_TIMESTAMP() AS _dbt_updated_at
FROM {{ ref('int_daily_order_metrics') }}
```

In [ ]:
# Mart models — materialized as TABLES (persisted, fast to query)

# --- fct_daily_sales ---
spark.sql("DROP TABLE IF EXISTS techmart_dw.fct_daily_sales")
spark.sql("""
CREATE TABLE techmart_dw.fct_daily_sales AS
SELECT
    order_date,
    total_orders,
    unique_customers,
    gross_revenue,
    completed_revenue,
    avg_order_value,
    CURRENT_TIMESTAMP() AS _dbt_updated_at
FROM techmart_dw.int_daily_order_metrics
""")

# --- dim_customers ---
spark.sql("DROP TABLE IF EXISTS techmart_dw.dim_customers")
spark.sql("""
CREATE TABLE techmart_dw.dim_customers AS
SELECT
    customer_id,
    full_name,
    email,
    customer_segment,
    lifetime_value,
    signed_up_at,
    total_orders,
    total_spent,
    avg_order_value,
    first_order_date,
    last_order_date,
    days_since_last_order,
    CASE
        WHEN days_since_last_order <= 30 THEN 'Active'
        WHEN days_since_last_order <= 90 THEN 'At Risk'
        WHEN days_since_last_order <= 180 THEN 'Lapsing'
        ELSE 'Churned'
    END AS activity_status,
    CURRENT_TIMESTAMP() AS _dbt_updated_at
FROM techmart_dw.int_customer_orders
""")

print("✅ Mart tables created (dbt equivalent: materialized='table'):")
mart_tables = ["fct_daily_sales", "dim_customers"]
for t in mart_tables:
    count = spark.table(f"techmart_dw.{t}").count()
    print(f"   {t}: {count:,} rows")

# Show the lineage
print("\n📊 Data Lineage (dbt DAG equivalent):")
print("   orders ──────────┐")
print("   order_items ─────┤")
print("   products ────────┤")
print("   customers ───────┤")
print("   payments ────────┘")
print("         │")
print("         ▼")
print("   stg_orders, stg_customers, stg_products, stg_order_items, stg_payments")
print("         │")
print("         ▼")
print("   int_order_items_enriched, int_customer_orders, int_daily_order_metrics")
print("         │")
print("         ▼")
print("   fct_daily_sales, dim_customers")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Build a Staging → Mart Pipeline
# ============================================
# Create the following dbt-style models using SQL:
#
# 1. stg_shipments (view) — from techmart_dw.shipments
#    - shipment_id, order_id, carrier (lowercased), 
#    - shipped_at (cast shipment_date to DATE), delivery_status (lowercased)
#
# 2. int_order_fulfillment (view) — join stg_orders + stg_shipments
#    - order_id, customer_id, order_total, order_status,
#    - carrier, delivery_status, days_to_ship (ordered_at to shipped_at)
#
# 3. fct_shipping_performance (table) — aggregate from int_order_fulfillment
#    - carrier, total_shipments, avg_days_to_ship, 
#    - on_time_rate (delivered within 5 days)
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
# 1. Staging
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.stg_shipments AS
SELECT
    shipment_id,
    order_id,
    LOWER(TRIM(carrier)) AS carrier,
    CAST(shipment_date AS DATE) AS shipped_at,
    LOWER(TRIM(delivery_status)) AS delivery_status
FROM techmart_dw.shipments
WHERE shipment_id IS NOT NULL
""")

# 2. Intermediate
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.int_order_fulfillment AS
SELECT
    o.order_id,
    o.customer_id,
    o.order_total,
    o.order_status,
    s.carrier,
    s.delivery_status,
    s.shipped_at,
    DATEDIFF(s.shipped_at, o.ordered_at) AS days_to_ship
FROM techmart_dw.stg_orders o
INNER JOIN techmart_dw.stg_shipments s ON o.order_id = s.order_id
""")

# 3. Mart
spark.sql("DROP TABLE IF EXISTS techmart_dw.fct_shipping_performance")
spark.sql("""
CREATE TABLE techmart_dw.fct_shipping_performance AS
SELECT
    carrier,
    COUNT(*) AS total_shipments,
    ROUND(AVG(days_to_ship), 1) AS avg_days_to_ship,
    ROUND(SUM(CASE WHEN days_to_ship <= 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS on_time_pct,
    CURRENT_TIMESTAMP() AS _dbt_updated_at
FROM techmart_dw.int_order_fulfillment
WHERE days_to_ship IS NOT NULL
GROUP BY carrier
""")

print("✅ Shipping pipeline built!")
spark.sql("SELECT * FROM techmart_dw.fct_shipping_performance ORDER BY total_shipments DESC").show()

---
# 📗 Section 4: Incremental Models

## What is Incremental Processing?

Instead of rebuilding the entire table every run, only process **new or changed** records.

```sql
-- models/marts/fct_daily_sales.sql (dbt incremental)
{{ config(
    materialized='incremental',
    unique_key='order_date',
    incremental_strategy='merge'
) }}

SELECT
    DATE_TRUNC('day', ordered_at) AS order_date,
    COUNT(*) AS total_orders,
    SUM(order_total) AS revenue
FROM {{ ref('stg_orders') }}

{% if is_incremental() %}
    -- Only process new data since last run
    WHERE ordered_at > (SELECT MAX(order_date) FROM {{ this }})
{% endif %}

GROUP BY 1
```

## Incremental Strategies

| Strategy | How | Best For |
|---|---|---|
| **Append** | INSERT new rows | Event logs, immutable data |
| **Merge** | MERGE on unique key | Fact tables with updates |
| **Delete+Insert** | Delete matching, insert new | When merge is complex |

## Key Concepts
- `{{ this }}` — refers to the current table (already materialized)
- `is_incremental()` — True when table already exists
- `unique_key` — column(s) to match for merge/upsert

In [ ]:
# Implement dbt-style incremental processing with MERGE
from datetime import datetime

class IncrementalModel:
    """Simulates dbt incremental materialization."""
    
    def __init__(self, target_table, unique_key, strategy="merge"):
        self.target_table = target_table
        self.unique_key = unique_key
        self.strategy = strategy
    
    def is_incremental(self):
        """Check if target table already exists (like dbt's is_incremental())."""
        try:
            spark.table(self.target_table)
            return True
        except:
            return False
    
    def get_max_watermark(self, column):
        """Get the max value of a column from the target (for filtering new data)."""
        if not self.is_incremental():
            return None
        result = spark.sql(f"SELECT MAX({column}) FROM {self.target_table}").collect()[0][0]
        return result
    
    def run(self, source_query, watermark_column=None):
        """Execute the incremental model."""
        print(f"🔄 Running incremental model: {self.target_table}")
        print(f"   Strategy: {self.strategy}")
        print(f"   Unique key: {self.unique_key}")
        
        if self.is_incremental() and watermark_column:
            max_val = self.get_max_watermark(watermark_column)
            print(f"   Mode: INCREMENTAL (table exists)")
            print(f"   Watermark: {watermark_column} > {max_val}")
            
            # Build incremental query
            if self.strategy == "merge":
                self._merge(source_query, watermark_column, max_val)
            elif self.strategy == "append":
                self._append(source_query, watermark_column, max_val)
        else:
            print(f"   Mode: FULL REFRESH (first run)")
            self._full_refresh(source_query)
    
    def _full_refresh(self, source_query):
        """First run: create table from scratch."""
        spark.sql(f"DROP TABLE IF EXISTS {self.target_table}")
        spark.sql(f"CREATE TABLE {self.target_table} AS {source_query}")
        count = spark.table(self.target_table).count()
        print(f"   ✅ Created with {count:,} rows")
    
    def _merge(self, source_query, watermark_col, max_val):
        """Merge new/updated records into target."""
        # Create temp view with new data
        spark.sql(f"""
            CREATE OR REPLACE TEMP VIEW _incremental_source AS
            {source_query}
            HAVING {watermark_col} > '{max_val}'
        """) if "GROUP BY" in source_query.upper() else spark.sql(f"""
            CREATE OR REPLACE TEMP VIEW _incremental_source AS
            SELECT * FROM ({source_query}) WHERE {watermark_col} > '{max_val}'
        """)
        
        new_count = spark.table("_incremental_source").count()
        print(f"   New records to merge: {new_count:,}")
        
        if new_count > 0:
            # Build MERGE statement
            merge_sql = f"""
                MERGE INTO {self.target_table} AS target
                USING _incremental_source AS source
                ON target.{self.unique_key} = source.{self.unique_key}
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """
            spark.sql(merge_sql)
            print(f"   ✅ Merged {new_count:,} records")
        else:
            print(f"   ⏭️ No new records to process")
    
    def _append(self, source_query, watermark_col, max_val):
        """Append-only: insert new records."""
        insert_sql = f"""
            INSERT INTO {self.target_table}
            SELECT * FROM ({source_query}) WHERE {watermark_col} > '{max_val}'
        """
        spark.sql(insert_sql)
        print(f"   ✅ Appended new records")


# Demo: Build an incremental daily sales model
print("=" * 60)
print("DEMO: Incremental Daily Sales (like dbt incremental)")
print("=" * 60)

# First run (full refresh)
sales_model = IncrementalModel(
    target_table="techmart_dw.inc_daily_sales",
    unique_key="order_date",
    strategy="merge"
)

source_sql = """
SELECT
    ordered_at AS order_date,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(order_total) AS total_revenue,
    AVG(order_total) AS avg_order_value
FROM techmart_dw.stg_orders
WHERE order_status IN ('completed', 'shipped')
GROUP BY ordered_at
"""

sales_model.run(source_sql, watermark_column="order_date")


In [ ]:
# Second run: simulate incremental (table already exists)
print("\n" + "=" * 60)
print("SECOND RUN: Incremental mode (only new data)")
print("=" * 60)

# The table now exists, so is_incremental() = True
sales_model2 = IncrementalModel(
    target_table="techmart_dw.inc_daily_sales",
    unique_key="order_date",
    strategy="merge"
)
sales_model2.run(source_sql, watermark_column="order_date")

# Show results
print("\n📊 inc_daily_sales (sample):")
spark.sql("""
    SELECT order_date, total_orders, total_revenue, avg_order_value
    FROM techmart_dw.inc_daily_sales
    ORDER BY order_date DESC
    LIMIT 10
""").show()


In [ ]:
# ============================================
# 🎯 YOUR TURN: Build an Incremental Model
# ============================================
# Create an incremental model for "customer activity":
#
# Target table: techmart_dw.inc_customer_activity
# Unique key: customer_id
# Strategy: merge
# Watermark: last_order_date
#
# Source query should produce:
#   customer_id, total_orders, total_spent, last_order_date, activity_status
#   (from stg_orders grouped by customer_id)
#
# 1. Use the IncrementalModel class
# 2. Run it once (full refresh)
# 3. Show the results
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
customer_activity_model = IncrementalModel(
    target_table="techmart_dw.inc_customer_activity",
    unique_key="customer_id",
    strategy="merge"
)

activity_sql = """
SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(order_total) AS total_spent,
    MAX(ordered_at) AS last_order_date,
    CASE
        WHEN DATEDIFF(CURRENT_DATE(), MAX(ordered_at)) <= 30 THEN 'Active'
        WHEN DATEDIFF(CURRENT_DATE(), MAX(ordered_at)) <= 90 THEN 'At Risk'
        ELSE 'Churned'
    END AS activity_status
FROM techmart_dw.stg_orders
GROUP BY customer_id
"""

customer_activity_model.run(activity_sql, watermark_column="last_order_date")

print("\n📊 Customer Activity (top 10 by spend):")
spark.sql("""
    SELECT customer_id, total_orders, total_spent, last_order_date, activity_status
    FROM techmart_dw.inc_customer_activity
    ORDER BY total_spent DESC
    LIMIT 10
""").show()


---
# 📗 Section 5: dbt Tests → Equivalent Quality Checks

## dbt Built-in Tests

dbt provides 4 generic tests out of the box:

| Test | What it Checks | SQL Equivalent |
|---|---|---|
| `not_null` | Column has no NULLs | `WHERE col IS NULL` → 0 rows |
| `unique` | Column has no duplicates | `GROUP BY col HAVING COUNT(*) > 1` → 0 rows |
| `accepted_values` | Column only has allowed values | `WHERE col NOT IN (...)` → 0 rows |
| `relationships` | FK exists in parent table | `LEFT JOIN ... WHERE parent.pk IS NULL` → 0 rows |

## How Tests are Defined in dbt

```yaml
# models/marts/_mart_models.yml
version: 2
models:
  - name: fct_daily_sales
    columns:
      - name: order_date
        tests:
          - not_null
          - unique
      - name: total_orders
        tests:
          - not_null
      - name: total_revenue
        tests:
          - not_null

  - name: dim_customers
    columns:
      - name: customer_id
        tests:
          - not_null
          - unique
      - name: activity_status
        tests:
          - accepted_values:
              values: ['Active', 'At Risk', 'Lapsing', 'Churned']
      - name: customer_id
        tests:
          - relationships:
              to: ref('stg_customers')
              field: customer_id
```

In [ ]:
# Implement dbt-style tests in Python/SQL

class DbtTestRunner:
    """Implements dbt-style data tests."""
    
    def __init__(self):
        self.results = []
    
    def not_null(self, table, column, severity="error"):
        """Test that a column has no NULL values."""
        count = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {table} WHERE {column} IS NULL
        """).collect()[0].cnt
        
        passed = count == 0
        self.results.append({
            "test": f"not_null_{table}_{column}",
            "table": table,
            "column": column,
            "type": "not_null",
            "passed": passed,
            "failures": count,
            "severity": severity
        })
        return passed
    
    def unique(self, table, column, severity="error"):
        """Test that a column has no duplicate values."""
        count = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM (
                SELECT {column}, COUNT(*) AS n
                FROM {table}
                GROUP BY {column}
                HAVING COUNT(*) > 1
            )
        """).collect()[0].cnt
        
        passed = count == 0
        self.results.append({
            "test": f"unique_{table}_{column}",
            "table": table,
            "column": column,
            "type": "unique",
            "passed": passed,
            "failures": count,
            "severity": severity
        })
        return passed
    
    def accepted_values(self, table, column, values, severity="error"):
        """Test that column only contains allowed values."""
        values_str = ", ".join([f"'{v}'" for v in values])
        count = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {table}
            WHERE {column} NOT IN ({values_str})
            AND {column} IS NOT NULL
        """).collect()[0].cnt
        
        passed = count == 0
        self.results.append({
            "test": f"accepted_values_{table}_{column}",
            "table": table,
            "column": column,
            "type": "accepted_values",
            "passed": passed,
            "failures": count,
            "severity": severity,
            "values": values
        })
        return passed
    
    def relationships(self, table, column, to_table, to_column, severity="error"):
        """Test referential integrity (FK exists in parent)."""
        count = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {table} child
            LEFT JOIN {to_table} parent ON child.{column} = parent.{to_column}
            WHERE parent.{to_column} IS NULL
            AND child.{column} IS NOT NULL
        """).collect()[0].cnt
        
        passed = count == 0
        self.results.append({
            "test": f"relationships_{table}_{column}_to_{to_table}",
            "table": table,
            "column": column,
            "type": "relationships",
            "passed": passed,
            "failures": count,
            "severity": severity
        })
        return passed
    
    def custom_test(self, name, query, severity="error"):
        """Run a custom SQL test (should return 0 rows to pass)."""
        count = spark.sql(f"SELECT COUNT(*) AS cnt FROM ({query})").collect()[0].cnt
        passed = count == 0
        self.results.append({
            "test": name,
            "type": "custom",
            "passed": passed,
            "failures": count,
            "severity": severity
        })
        return passed
    
    def summary(self):
        """Print test results summary (like dbt test output)."""
        passed = sum(1 for r in self.results if r["passed"])
        failed = sum(1 for r in self.results if not r["passed"])
        warns = sum(1 for r in self.results if not r["passed"] and r["severity"] == "warn")
        errors = sum(1 for r in self.results if not r["passed"] and r["severity"] == "error")
        
        print(f"\n{'='*60}")
        print(f"  dbt Test Results: {passed} passed, {failed} failed")
        if warns: print(f"  ⚠️  Warnings: {warns}")
        if errors: print(f"  ❌ Errors: {errors}")
        print(f"{'='*60}")
        
        for r in self.results:
            icon = "✅ PASS" if r["passed"] else ("⚠️ WARN" if r["severity"] == "warn" else "❌ FAIL")
            print(f"  {icon} | {r['test']}")
            if not r["passed"]:
                print(f"         Failures: {r['failures']} records")
        
        print(f"{'='*60}")
        return errors == 0  # Only errors block deployment


# Run tests on our mart models
tester = DbtTestRunner()

print("🧪 Running dbt-style tests on fct_daily_sales...")
tester.not_null("techmart_dw.fct_daily_sales", "order_date")
tester.unique("techmart_dw.fct_daily_sales", "order_date")
tester.not_null("techmart_dw.fct_daily_sales", "total_orders")
tester.not_null("techmart_dw.fct_daily_sales", "total_revenue")

print("🧪 Running dbt-style tests on dim_customers...")
tester.not_null("techmart_dw.dim_customers", "customer_id")
tester.unique("techmart_dw.dim_customers", "customer_id")
tester.accepted_values("techmart_dw.dim_customers", "activity_status",
                       ["Active", "At Risk", "Lapsing", "Churned"])
tester.relationships("techmart_dw.dim_customers", "customer_id",
                     "techmart_dw.stg_customers", "customer_id")

# Custom test: revenue should always be positive
tester.custom_test(
    "positive_revenue",
    "SELECT * FROM techmart_dw.fct_daily_sales WHERE total_revenue < 0"
)

tester.summary()


In [ ]:
# ============================================
# 🎯 YOUR TURN: Write dbt-Style Tests
# ============================================
# Write tests for the fct_shipping_performance table:
#
# 1. not_null on: carrier, total_shipments
# 2. unique on: carrier
# 3. accepted_values on carrier: check what carriers exist first,
#    then test against those values
# 4. Custom test: avg_days_to_ship should be between 0 and 30
# 5. Custom test: on_time_pct should be between 0 and 100
#
# Use the DbtTestRunner class and print the summary.
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
# First, see what carriers we have
carriers = [row.carrier for row in 
    spark.sql("SELECT DISTINCT carrier FROM techmart_dw.fct_shipping_performance").collect()]
print(f"Carriers found: {carriers}")

# Run tests
ship_tester = DbtTestRunner()

ship_tester.not_null("techmart_dw.fct_shipping_performance", "carrier")
ship_tester.not_null("techmart_dw.fct_shipping_performance", "total_shipments")
ship_tester.unique("techmart_dw.fct_shipping_performance", "carrier")
ship_tester.accepted_values("techmart_dw.fct_shipping_performance", "carrier", carriers)

# Custom: avg_days_to_ship between 0 and 30
ship_tester.custom_test(
    "valid_avg_days_to_ship",
    """SELECT * FROM techmart_dw.fct_shipping_performance 
       WHERE avg_days_to_ship < 0 OR avg_days_to_ship > 30"""
)

# Custom: on_time_pct between 0 and 100
ship_tester.custom_test(
    "valid_on_time_pct",
    """SELECT * FROM techmart_dw.fct_shipping_performance 
       WHERE on_time_pct < 0 OR on_time_pct > 100"""
)

ship_tester.summary()


---
# 📗 Section 6: dbt Sources & Documentation

## Source Freshness

dbt can check if source data is fresh (not stale):

```yaml
sources:
  - name: raw_techmart
    tables:
      - name: orders
        loaded_at_field: _loaded_at
        freshness:
          warn_after: {count: 12, period: hour}
          error_after: {count: 24, period: hour}
```

## Column Documentation

```yaml
models:
  - name: dim_customers
    description: "Customer dimension with activity metrics"
    columns:
      - name: customer_id
        description: "Unique customer identifier"
      - name: activity_status
        description: "Current activity classification based on recency"
        tests:
          - accepted_values:
              values: ['Active', 'At Risk', 'Lapsing', 'Churned']
```

In [ ]:
# Implement source freshness checks (dbt-style)
from datetime import datetime, timedelta

class SourceFreshnessChecker:
    """Check if source data is fresh (like dbt source freshness)."""
    
    def __init__(self):
        self.results = []
    
    def check(self, table, loaded_at_column, warn_hours=12, error_hours=24):
        """Check freshness of a source table."""
        try:
            max_loaded = spark.sql(f"""
                SELECT MAX({loaded_at_column}) AS max_ts FROM {table}
            """).collect()[0].max_ts
            
            if max_loaded is None:
                status = "error"
                hours_ago = None
                message = "No data found"
            else:
                now = datetime.now()
                # Handle both date and timestamp
                if hasattr(max_loaded, 'hour'):
                    hours_ago = (now - max_loaded).total_seconds() / 3600
                else:
                    hours_ago = (now.date() - max_loaded).days * 24
                
                if hours_ago > error_hours:
                    status = "error"
                elif hours_ago > warn_hours:
                    status = "warn"
                else:
                    status = "pass"
                message = f"Last loaded {hours_ago:.0f} hours ago"
        except Exception as e:
            status = "error"
            hours_ago = None
            message = str(e)
        
        self.results.append({
            "table": table,
            "column": loaded_at_column,
            "status": status,
            "hours_ago": hours_ago,
            "message": message,
            "warn_threshold": warn_hours,
            "error_threshold": error_hours
        })
    
    def summary(self):
        """Print freshness results."""
        print(f"\n{'='*70}")
        print(f"  Source Freshness Results")
        print(f"{'='*70}")
        print(f"  {'Table':<35} {'Status':<10} {'Age':<15} {'Thresholds'}")
        print(f"  {'-'*70}")
        for r in self.results:
            icon = {"pass": "✅", "warn": "⚠️", "error": "❌"}[r["status"]]
            age = f"{r['hours_ago']:.0f}h ago" if r['hours_ago'] else "N/A"
            thresholds = f"warn>{r['warn_threshold']}h, err>{r['error_threshold']}h"
            print(f"  {icon} {r['table']:<33} {r['status']:<10} {age:<15} {thresholds}")
        print(f"{'='*70}")


# Check freshness of our sources
freshness = SourceFreshnessChecker()

# These use order_date as a proxy for loaded_at
freshness.check("techmart_dw.orders", "order_date", warn_hours=720, error_hours=1440)
freshness.check("techmart_dw.customers", "signup_date", warn_hours=720, error_hours=1440)
freshness.check("techmart_dw.payments", "payment_date", warn_hours=720, error_hours=1440)
freshness.check("techmart_dw.shipments", "shipment_date", warn_hours=720, error_hours=1440)

freshness.summary()
print("\nNote: Using generous thresholds since this is static demo data.")


In [ ]:
# dbt-style documentation catalog
class ModelDocumentation:
    """Generate dbt-style documentation for models."""
    
    def __init__(self):
        self.models = []
    
    def document_model(self, table_name, description, column_docs=None):
        """Document a model with column descriptions."""
        df = spark.table(table_name)
        columns = []
        for field in df.schema.fields:
            col_doc = {
                "name": field.name,
                "type": str(field.dataType).replace("()", ""),
                "description": (column_docs or {}).get(field.name, "")
            }
            columns.append(col_doc)
        
        self.models.append({
            "name": table_name.split(".")[-1],
            "full_name": table_name,
            "description": description,
            "columns": columns,
            "row_count": df.count()
        })
    
    def generate_catalog(self):
        """Print documentation catalog (like dbt docs generate)."""
        print("📚 DATA CATALOG (dbt docs equivalent)")
        print("=" * 70)
        for model in self.models:
            print(f"\n┌{'─'*68}┐")
            print(f"│ 📋 {model['name']:<64}│")
            print(f"├{'─'*68}┤")
            print(f"│ {model['description']:<67}│")
            print(f"│ Rows: {model['row_count']:,}{'':<58}│")
            print(f"├{'─'*68}┤")
            print(f"│ {'Column':<25} {'Type':<20} {'Description':<22}│")
            print(f"│ {'─'*25} {'─'*20} {'─'*22}│")
            for col in model['columns']:
                desc = col['description'][:22] if col['description'] else ""
                print(f"│ {col['name']:<25} {col['type']:<20} {desc:<22}│")
            print(f"└{'─'*68}┘")


# Document our mart models
docs = ModelDocumentation()

docs.document_model("techmart_dw.fct_daily_sales",
    "Daily sales aggregates for business reporting",
    {
        "order_date": "Calendar date of orders",
        "total_orders": "Count of distinct orders",
        "unique_customers": "Count of distinct buyers",
        "gross_revenue": "Sum of all order totals",
        "completed_revenue": "Revenue from completed orders",
        "avg_order_value": "Average order amount",
        "_dbt_updated_at": "When this row was last updated"
    })

docs.document_model("techmart_dw.dim_customers",
    "Customer dimension with lifetime metrics and activity status",
    {
        "customer_id": "Unique customer identifier (PK)",
        "full_name": "First + last name",
        "email": "Normalized email address",
        "customer_segment": "Business segment classification",
        "lifetime_value": "Total historical spend",
        "total_orders": "Lifetime order count",
        "activity_status": "Active/At Risk/Lapsing/Churned",
        "_dbt_updated_at": "When this row was last updated"
    })

docs.generate_catalog()


---
# 📗 Section 7: Jinja & Macros → Python Equivalents

## Jinja in dbt

dbt uses Jinja templating to make SQL dynamic:

```sql
-- macros/cents_to_dollars.sql
{% macro cents_to_dollars(column_name, precision=2) %}
    ROUND({{ column_name }} / 100.0, {{ precision }})
{% endmacro %}

-- Usage in a model:
SELECT
    order_id,
    {{ cents_to_dollars('amount_cents') }} AS amount_dollars
FROM {{ ref('stg_orders') }}
```

## Control Flow in Jinja

```sql
-- Dynamic column selection
{% set payment_methods = ['credit_card', 'debit_card', 'paypal', 'bank_transfer'] %}

SELECT
    order_id,
    {% for method in payment_methods %}
        SUM(CASE WHEN payment_method = '{{ method }}' THEN amount ELSE 0 END) 
            AS {{ method }}_amount
        {% if not loop.last %},{% endif %}
    {% endfor %}
FROM {{ ref('stg_payments') }}
GROUP BY order_id
```

## Python Equivalent

In Python, we use **f-strings and functions** to achieve the same thing.

In [ ]:
# Python equivalents of dbt Jinja macros

# --- Macro: cents_to_dollars ---
def cents_to_dollars(column_name, precision=2):
    """dbt macro equivalent: convert cents to dollars."""
    return f"ROUND({column_name} / 100.0, {precision})"

# --- Macro: generate_surrogate_key ---
def generate_surrogate_key(*columns):
    """dbt macro equivalent: create a surrogate key from multiple columns."""
    concat_expr = " || '-' || ".join([f"COALESCE(CAST({col} AS STRING), '')" for col in columns])
    return f"MD5({concat_expr})"

# --- Macro: star (select all except) ---
def star(table, except_columns=None):
    """dbt macro equivalent: select all columns except specified ones."""
    df = spark.table(table)
    cols = [c for c in df.columns if c not in (except_columns or [])]
    return ", ".join(cols)

# --- Macro: pivot ---
def pivot(column, values, agg_func="SUM", value_column="amount"):
    """dbt macro equivalent: pivot a column into multiple columns."""
    cases = []
    for val in values:
        safe_name = val.replace(" ", "_").replace("-", "_").lower()
        cases.append(
            f"{agg_func}(CASE WHEN {column} = '{val}' THEN {value_column} ELSE 0 END) AS {safe_name}_{value_column}"
        )
    return ",
    ".join(cases)

# --- Macro: date_spine ---
def date_spine(start_date, end_date):
    """dbt macro equivalent: generate a date range."""
    return f"SELECT EXPLODE(SEQUENCE(DATE('{start_date}'), DATE('{end_date}'), INTERVAL 1 DAY)) AS date_day"


# Demo: Use macros to build queries
print("📝 Macro: cents_to_dollars")
print(f"   SQL: SELECT {cents_to_dollars('price_cents')}")

print("\n📝 Macro: generate_surrogate_key")
print(f"   SQL: {generate_surrogate_key('order_id', 'product_id')}")

print("\n📝 Macro: star (select all except)")
print(f"   SQL: SELECT {star('techmart_dw.stg_orders', except_columns=['shipping_address'])}")

print("\n📝 Macro: pivot")
payment_methods = ["credit_card", "debit_card", "paypal", "bank_transfer"]
print(f"   SQL:\n    {pivot('payment_method', payment_methods, 'SUM', 'payment_amount')}")

print("\n📝 Macro: date_spine")
print(f"   SQL: {date_spine('2024-01-01', '2024-12-31')}")


In [ ]:
# Use macros in a real query — pivot payments by method
payment_methods = ["credit_card", "debit_card", "paypal", "bank_transfer"]

pivot_query = f"""
SELECT
    o.order_id,
    o.order_total,
    {pivot('p.payment_method', payment_methods, 'SUM', 'p.payment_amount')}
FROM techmart_dw.stg_orders o
LEFT JOIN techmart_dw.stg_payments p ON o.order_id = p.order_id
GROUP BY o.order_id, o.order_total
LIMIT 10
"""

print("📝 Generated pivot query:")
print(pivot_query)
print("\n📊 Results:")
spark.sql(pivot_query).show()


In [ ]:
# Advanced: Model builder using macros (like a mini dbt)
class ModelBuilder:
    """Build SQL models using macro functions (Python dbt equivalent)."""
    
    def __init__(self, target_schema="techmart_dw"):
        self.schema = target_schema
        self.models = {}
    
    def ref(self, model_name):
        """dbt ref() equivalent — resolve model to table name."""
        return f"{self.schema}.{model_name}"
    
    def source(self, source_name, table_name):
        """dbt source() equivalent — resolve source to table name."""
        return f"{self.schema}.{table_name}"
    
    def add_model(self, name, sql, materialization="view"):
        """Register a model."""
        self.models[name] = {"sql": sql, "materialization": materialization}
    
    def build(self, model_name):
        """Build (materialize) a model."""
        model = self.models[model_name]
        full_name = f"{self.schema}.{model_name}"
        
        if model["materialization"] == "view":
            spark.sql(f"CREATE OR REPLACE VIEW {full_name} AS {model['sql']}")
        elif model["materialization"] == "table":
            spark.sql(f"DROP TABLE IF EXISTS {full_name}")
            spark.sql(f"CREATE TABLE {full_name} AS {model['sql']}")
        
        count = spark.table(full_name).count()
        print(f"  ✅ Built {model_name} ({model['materialization']}): {count:,} rows")
    
    def build_all(self):
        """Build all registered models."""
        print(f"🏗️ Building {len(self.models)} models...")
        for name in self.models:
            self.build(name)
        print("  Done!")


# Use the model builder
mb = ModelBuilder()

# Register models using macros
mb.add_model("macro_demo_payments_pivot", f"""
    SELECT
        order_id,
        {pivot('payment_method', ['credit_card', 'debit_card', 'paypal', 'bank_transfer'], 'SUM', 'payment_amount')}
    FROM {mb.ref('stg_payments')}
    GROUP BY order_id
""", materialization="view")

mb.add_model("macro_demo_order_keys", f"""
    SELECT
        {generate_surrogate_key('order_id', 'customer_id')} AS order_sk,
        order_id,
        customer_id,
        order_total
    FROM {mb.ref('stg_orders')}
""", materialization="view")

mb.build_all()

# Show results
print("\n📊 Payments pivot (first 5):")
spark.sql(f"SELECT * FROM {mb.ref('macro_demo_payments_pivot')} LIMIT 5").show(truncate=False)


In [ ]:
# ============================================
# 🎯 YOUR TURN: Build Reusable SQL Macros
# ============================================
# Create these Python macros (dbt Jinja equivalents):
#
# 1. safe_divide(numerator, denominator, default=0)
#    → Returns: "COALESCE(numerator / NULLIF(denominator, 0), default)"
#
# 2. date_trunc_to(column, grain)  where grain is 'day', 'week', 'month'
#    → Returns: "DATE_TRUNC('grain', column)"
#
# 3. classify_tier(column, thresholds_dict)
#    → Returns a CASE WHEN statement based on thresholds
#    Example: classify_tier('amount', {'high': 5000, 'medium': 1000})
#    → "CASE WHEN amount > 5000 THEN 'high' WHEN amount > 1000 THEN 'medium' ELSE 'low' END"
#
# Then use all 3 macros to build a query against stg_orders and run it.
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
def safe_divide(numerator, denominator, default=0):
    """Safely divide, returning default on division by zero."""
    return f"COALESCE({numerator} / NULLIF({denominator}, 0), {default})"

def date_trunc_to(column, grain):
    """Truncate a date/timestamp to specified grain."""
    return f"DATE_TRUNC('{grain}', {column})"

def classify_tier(column, thresholds):
    """Generate CASE WHEN for tier classification."""
    # Sort thresholds descending
    sorted_thresholds = sorted(thresholds.items(), key=lambda x: x[1], reverse=True)
    cases = [f"WHEN {column} > {val} THEN '{name}'" for name, val in sorted_thresholds]
    return f"CASE {' '.join(cases)} ELSE 'low' END"

# Use all 3 macros in a query
query = f"""
SELECT
    {date_trunc_to('ordered_at', 'month')} AS order_month,
    COUNT(*) AS total_orders,
    SUM(order_total) AS revenue,
    {safe_divide('SUM(order_total)', 'COUNT(*)')} AS avg_order_value,
    {classify_tier('SUM(order_total)', {'high': 500000, 'medium': 100000})} AS month_tier
FROM techmart_dw.stg_orders
GROUP BY {date_trunc_to('ordered_at', 'month')}
ORDER BY order_month DESC
LIMIT 12
"""

print("📝 Generated query using macros:")
print(query)
print("\n📊 Results:")
spark.sql(query).show()


---
# 📗 Section 8: dbt on Databricks

## Integration Options

| Option | Description | Best For |
|---|---|---|
| **dbt Core + dbt-databricks** | Open source, CLI-based | Teams with existing dbt skills |
| **dbt Cloud** | Managed SaaS platform | Teams wanting managed IDE + scheduling |
| **Databricks SQL + Notebooks** | Native Databricks | Teams already on Databricks |
| **Lakeflow Declarative Pipelines** | Databricks-native (formerly DLT) | Streaming + batch in one framework |

## dbt-databricks Adapter

```bash
# Install
pip install dbt-databricks

# profiles.yml (connection config)
techmart:
  target: dev
  outputs:
    dev:
      type: databricks
      catalog: dev_catalog
      schema: techmart_dev
      host: dbc-12345.cloud.databricks.com
      http_path: /sql/1.0/warehouses/abc123
      token: "{{ env_var('DATABRICKS_TOKEN') }}"
      threads: 4
    prod:
      type: databricks
      catalog: prod_catalog
      schema: techmart_prod
      host: dbc-12345.cloud.databricks.com
      http_path: /sql/1.0/warehouses/xyz789
      token: "{{ env_var('DATABRICKS_TOKEN') }}"
      threads: 8
```

## dbt Commands

```bash
dbt run              # Build all models
dbt run --select staging   # Build only staging models
dbt test             # Run all tests
dbt docs generate    # Generate documentation
dbt docs serve       # Serve docs locally
dbt source freshness # Check source freshness
dbt build            # run + test in dependency order
```

## dbt vs Lakeflow Declarative Pipelines (formerly DLT)

| Feature | dbt | Lakeflow Declarative Pipelines |
|---|---|---|
| Language | SQL (+ Jinja) | SQL or Python |
| Streaming | ❌ Batch only | ✅ Streaming + batch |
| Expectations | Via tests (post-hoc) | Built-in (inline) |
| Orchestration | External (Airflow, etc.) | Built-in |
| Lineage | dbt-generated DAG | Unity Catalog lineage |
| Incremental | Merge/append strategies | Auto-managed |
| Best for | SQL-heavy analytics teams | Full platform teams |

In [ ]:
# Comparison: dbt model vs Lakeflow Declarative Pipeline vs raw SQL

print("""
╔══════════════════════════════════════════════════════════════════════╗
║  SAME LOGIC, THREE APPROACHES                                       ║
╚══════════════════════════════════════════════════════════════════════╝

━━━ 1. dbt Model (models/marts/fct_daily_sales.sql) ━━━

{{ config(materialized='incremental', unique_key='order_date') }}

SELECT
    DATE_TRUNC('day', ordered_at) AS order_date,
    COUNT(*) AS total_orders,
    SUM(order_total) AS revenue
FROM {{ ref('stg_orders') }}
{% if is_incremental() %}
WHERE ordered_at > (SELECT MAX(order_date) FROM {{ this }})
{% endif %}
GROUP BY 1

━━━ 2. Lakeflow Declarative Pipeline (Python) ━━━

import dlt
from pyspark.sql.functions import *

@dlt.table(
    comment="Daily sales aggregates",
    table_properties={"quality": "gold"}
)
@dlt.expect_or_drop("positive_revenue", "revenue > 0")
def fct_daily_sales():
    return (
        dlt.read("stg_orders")
        .groupBy(date_trunc("day", "ordered_at").alias("order_date"))
        .agg(
            count("*").alias("total_orders"),
            sum("order_total").alias("revenue")
        )
    )

━━━ 3. Raw SQL (what we do in this notebook) ━━━

CREATE OR REPLACE TABLE techmart_dw.fct_daily_sales AS
SELECT
    DATE_TRUNC('day', ordered_at) AS order_date,
    COUNT(*) AS total_orders,
    SUM(order_total) AS revenue
FROM techmart_dw.stg_orders
GROUP BY 1;

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

KEY INSIGHT: The SQL logic is identical across all three.
The difference is in orchestration, testing, and deployment.
""")


---
# 📗 Section 9: Mini Projects

## Project 1: Complete dbt-Style Transformation Layer

Build a full transformation layer following dbt conventions:
- 5 staging models
- 3 intermediate models
- 2 mart models
- Tests for every model
- Documentation

In [ ]:
# PROJECT 1: Complete dbt-style transformation layer
# We already built staging + intermediate + marts above.
# Now let's add a comprehensive test suite and documentation.

class DbtProject:
    """Simulates a complete dbt project execution."""
    
    def __init__(self, name, schema="techmart_dw"):
        self.name = name
        self.schema = schema
        self.models = {}
        self.tests = []
        self.docs = {}
    
    def add_staging(self, name, sql):
        self.models[name] = {"sql": sql, "layer": "staging", "materialization": "view"}
    
    def add_intermediate(self, name, sql):
        self.models[name] = {"sql": sql, "layer": "intermediate", "materialization": "view"}
    
    def add_mart(self, name, sql):
        self.models[name] = {"sql": sql, "layer": "marts", "materialization": "table"}
    
    def add_test(self, test_type, table, column=None, **kwargs):
        self.tests.append({"type": test_type, "table": table, "column": column, **kwargs})
    
    def add_doc(self, model_name, description, columns=None):
        self.docs[model_name] = {"description": description, "columns": columns or {}}
    
    def run(self):
        """Execute all models in dependency order."""
        print(f"\n🏗️ dbt run — {self.name}")
        print(f"{'='*60}")
        
        # Build in order: staging → intermediate → marts
        for layer in ["staging", "intermediate", "marts"]:
            layer_models = {k: v for k, v in self.models.items() if v["layer"] == layer}
            if layer_models:
                print(f"\n  📂 {layer}/")
                for name, model in layer_models.items():
                    full_name = f"{self.schema}.{name}"
                    try:
                        if model["materialization"] == "view":
                            spark.sql(f"CREATE OR REPLACE VIEW {full_name} AS {model['sql']}")
                        else:
                            spark.sql(f"DROP TABLE IF EXISTS {full_name}")
                            spark.sql(f"CREATE TABLE {full_name} AS {model['sql']}")
                        count = spark.table(full_name).count()
                        print(f"    ✅ {name} ({model['materialization']}): {count:,} rows")
                    except Exception as e:
                        print(f"    ❌ {name}: {e}")
    
    def test(self):
        """Run all tests."""
        print(f"\n🧪 dbt test — {self.name}")
        print(f"{'='*60}")
        
        tester = DbtTestRunner()
        for t in self.tests:
            full_table = f"{self.schema}.{t['table']}"
            if t["type"] == "not_null":
                tester.not_null(full_table, t["column"])
            elif t["type"] == "unique":
                tester.unique(full_table, t["column"])
            elif t["type"] == "accepted_values":
                tester.accepted_values(full_table, t["column"], t["values"])
            elif t["type"] == "relationships":
                tester.relationships(full_table, t["column"],
                    f"{self.schema}.{t['to']}", t["field"])
        
        return tester.summary()
    
    def docs_generate(self):
        """Generate documentation."""
        print(f"\n📚 dbt docs generate — {self.name}")
        print(f"{'='*60}")
        for model_name, doc in self.docs.items():
            print(f"\n  📋 {model_name}: {doc['description']}")
            if doc['columns']:
                for col, desc in doc['columns'].items():
                    print(f"     • {col}: {desc}")


# Build the project
project = DbtProject("techmart_analytics")

# Staging (already created, but register them)
project.add_staging("stg_orders", """
    SELECT order_id, customer_id, CAST(total_amount AS DECIMAL(12,2)) AS order_total,
           LOWER(TRIM(status)) AS order_status, CAST(order_date AS DATE) AS ordered_at,
           order_source, shipping_address
    FROM techmart_dw.orders WHERE order_id IS NOT NULL
""")

project.add_staging("stg_customers", """
    SELECT customer_id, LOWER(TRIM(email)) AS email, first_name, last_name,
           CONCAT(first_name, ' ', last_name) AS full_name, customer_segment,
           CAST(lifetime_value AS DECIMAL(12,2)) AS lifetime_value,
           CAST(signup_date AS DATE) AS signed_up_at, is_active
    FROM techmart_dw.customers WHERE customer_id IS NOT NULL
""")

# Intermediate
project.add_intermediate("int_customer_lifetime", """
    SELECT c.customer_id, c.full_name, c.customer_segment,
           COUNT(DISTINCT o.order_id) AS lifetime_orders,
           SUM(o.order_total) AS lifetime_revenue,
           AVG(o.order_total) AS avg_order_value,
           DATEDIFF(MAX(o.ordered_at), MIN(o.ordered_at)) AS customer_tenure_days
    FROM techmart_dw.stg_customers c
    LEFT JOIN techmart_dw.stg_orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.full_name, c.customer_segment
""")

# Mart
project.add_mart("fct_customer_cohorts", """
    SELECT
        DATE_TRUNC('month', c.signed_up_at) AS cohort_month,
        c.customer_segment,
        COUNT(DISTINCT c.customer_id) AS cohort_size,
        SUM(CASE WHEN cl.lifetime_orders > 0 THEN 1 ELSE 0 END) AS converted_customers,
        AVG(cl.lifetime_revenue) AS avg_lifetime_revenue,
        AVG(cl.lifetime_orders) AS avg_lifetime_orders
    FROM techmart_dw.stg_customers c
    LEFT JOIN techmart_dw.int_customer_lifetime cl ON c.customer_id = cl.customer_id
    GROUP BY DATE_TRUNC('month', c.signed_up_at), c.customer_segment
""")

# Tests
project.add_test("not_null", "fct_customer_cohorts", "cohort_month")
project.add_test("not_null", "fct_customer_cohorts", "cohort_size")
project.add_test("not_null", "int_customer_lifetime", "customer_id")
project.add_test("unique", "int_customer_lifetime", "customer_id")

# Documentation
project.add_doc("fct_customer_cohorts", "Monthly customer cohort analysis", {
    "cohort_month": "Month the customer signed up",
    "cohort_size": "Number of customers in this cohort",
    "converted_customers": "Customers who placed at least 1 order",
    "avg_lifetime_revenue": "Average total spend per customer in cohort"
})

# Execute the full dbt workflow
project.run()
project.test()
project.docs_generate()


## Project 2: Incremental Pipeline with Quality Tests

In [ ]:
# PROJECT 2: Incremental pipeline with full quality suite
print("=" * 60)
print("PROJECT 2: Incremental Order Processing Pipeline")
print("=" * 60)

# Step 1: Create the incremental target table
spark.sql("DROP TABLE IF EXISTS techmart_dw.inc_order_summary")
spark.sql("""
    CREATE TABLE techmart_dw.inc_order_summary (
        order_date DATE,
        order_source STRING,
        total_orders INT,
        total_revenue DECIMAL(14,2),
        avg_order_value DECIMAL(10,2),
        unique_customers INT,
        _loaded_at TIMESTAMP
    )
""")
print("✅ Created target table: inc_order_summary")

# Step 2: Initial load (full refresh)
spark.sql("""
    INSERT INTO techmart_dw.inc_order_summary
    SELECT
        ordered_at AS order_date,
        order_source,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(order_total) AS total_revenue,
        AVG(order_total) AS avg_order_value,
        COUNT(DISTINCT customer_id) AS unique_customers,
        CURRENT_TIMESTAMP() AS _loaded_at
    FROM techmart_dw.stg_orders
    WHERE order_status IN ('completed', 'shipped')
    GROUP BY ordered_at, order_source
""")
initial_count = spark.table("techmart_dw.inc_order_summary").count()
print(f"✅ Initial load: {initial_count:,} rows")

# Step 3: Simulate incremental load (MERGE)
print("\n🔄 Simulating incremental update...")
spark.sql("""
    MERGE INTO techmart_dw.inc_order_summary AS target
    USING (
        SELECT
            ordered_at AS order_date,
            order_source,
            COUNT(DISTINCT order_id) AS total_orders,
            SUM(order_total) AS total_revenue,
            AVG(order_total) AS avg_order_value,
            COUNT(DISTINCT customer_id) AS unique_customers,
            CURRENT_TIMESTAMP() AS _loaded_at
        FROM techmart_dw.stg_orders
        WHERE order_status IN ('completed', 'shipped')
          AND ordered_at >= (SELECT MAX(order_date) FROM techmart_dw.inc_order_summary)
        GROUP BY ordered_at, order_source
    ) AS source
    ON target.order_date = source.order_date 
       AND target.order_source = source.order_source
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
final_count = spark.table("techmart_dw.inc_order_summary").count()
print(f"✅ After incremental: {final_count:,} rows")

# Step 4: Run quality tests
print("\n🧪 Running quality tests...")
tester = DbtTestRunner()
tester.not_null("techmart_dw.inc_order_summary", "order_date")
tester.not_null("techmart_dw.inc_order_summary", "total_orders")
tester.not_null("techmart_dw.inc_order_summary", "total_revenue")
tester.custom_test("positive_orders", 
    "SELECT * FROM techmart_dw.inc_order_summary WHERE total_orders <= 0")
tester.custom_test("positive_revenue",
    "SELECT * FROM techmart_dw.inc_order_summary WHERE total_revenue < 0")
tester.custom_test("reasonable_avg",
    "SELECT * FROM techmart_dw.inc_order_summary WHERE avg_order_value > 50000")
tester.summary()

# Step 5: Show sample results
print("\n📊 Sample output:")
spark.sql("""
    SELECT order_date, order_source, total_orders, total_revenue, avg_order_value
    FROM techmart_dw.inc_order_summary
    ORDER BY order_date DESC, total_revenue DESC
    LIMIT 10
""").show()


---
# 📗 Section 10: Interview Questions

In [ ]:
# Interview Question 1
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q1: What is dbt and how does it fit in the modern       ║
║  data stack?                                                        ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"dbt (data build tool) is a SQL-first transformation framework that
implements the T in ELT. It fits between ingestion tools (Fivetran,
Airbyte) and BI tools (Looker, Tableau).

KEY VALUE PROPOSITIONS:
1. Write SELECT statements → dbt handles CREATE TABLE/VIEW
2. Automatic dependency management via ref() function
3. Built-in testing (not_null, unique, accepted_values, relationships)
4. Auto-generated documentation and lineage graphs
5. Environment management (dev/staging/prod from same code)
6. Version control friendly (SQL files in Git)

WHERE IT FITS:
  Sources → Ingestion (Fivetran) → Raw Layer → dbt → Analytics Layer → BI

WHY IT'S POPULAR:
- Analytics engineers can work in SQL (no Python needed)
- Software engineering practices for data (Git, CI/CD, testing)
- Modular, reusable transformations
- Self-documenting data warehouse"
""")


In [ ]:
# Interview Question 2
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q2: Explain dbt materializations                        ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"dbt supports 4 materialization strategies:

1. VIEW (default for staging):
   - Creates a SQL view (no data stored)
   - Always fresh (queries source on read)
   - Fast to build, slow to query at scale
   - Use for: staging models, light transforms

2. TABLE (default for marts):
   - Creates a physical table (data stored)
   - Fast to query, but full rebuild each run
   - Use for: final mart tables, aggregations

3. INCREMENTAL (for large fact tables):
   - Only processes new/changed records
   - Uses MERGE or INSERT for updates
   - Requires unique_key and is_incremental() logic
   - Use for: event tables, large fact tables

4. EPHEMERAL (for intermediate logic):
   - Not materialized at all — injected as CTE
   - Zero storage cost, but can't be queried directly
   - Use for: intermediate calculations, DRY logic

DECISION FRAMEWORK:
  - Small + frequently changing → VIEW
  - Large + queried often → TABLE
  - Very large + append-mostly → INCREMENTAL
  - Reusable logic, not queried directly → EPHEMERAL"
""")


In [ ]:
# Interview Question 3
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q3: How does dbt handle dependencies between models?    ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"dbt uses the ref() function to create an explicit dependency graph (DAG):

  SELECT * FROM {{ ref('stg_orders') }}

When dbt sees ref('stg_orders'), it:
1. Resolves the model name to the actual table/view name
2. Adds an edge in the dependency graph
3. Ensures stg_orders is built BEFORE any model that references it
4. Handles environment-specific naming (dev vs prod schemas)

EXECUTION ORDER:
  dbt builds a DAG and executes in topological order:
  
  stg_orders ──┐
  stg_items ───┤──▶ int_order_enriched ──▶ fct_daily_sales
  stg_products─┘

BENEFITS:
- No manual ordering of SQL scripts
- Parallel execution of independent models
- Clear lineage visualization
- Prevents circular dependencies (dbt errors on cycles)
- Selective execution: dbt run --select fct_daily_sales+
  (runs the model AND all its upstream dependencies)"
""")


In [ ]:
# Interview Question 4
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q4: What testing capabilities does dbt provide?         ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"dbt provides two categories of tests:

1. GENERIC (SCHEMA) TESTS — defined in YAML:
   - not_null: ensures no NULL values
   - unique: ensures no duplicates
   - accepted_values: column only has allowed values
   - relationships: referential integrity (FK exists)
   
   These are defined per-column in schema.yml files.

2. SINGULAR (CUSTOM) TESTS — SQL files in tests/:
   - Any SELECT that returns rows = test failure
   - Example: SELECT * FROM orders WHERE amount < 0
   - For complex business rules that don't fit generic tests

3. PACKAGES extend testing:
   - dbt_expectations: Great Expectations-style tests
   - dbt_utils: recency, equal_rowcount, expression_is_true
   - elementary: anomaly detection, schema changes

TEST SEVERITY:
   - error: blocks deployment (default)
   - warn: logs warning but continues

WHEN TESTS RUN:
   - dbt test: run all tests
   - dbt build: run + test in dependency order
   - CI/CD: tests gate deployment to production

BEST PRACTICE:
   Every model should have at minimum:
   - not_null on primary key
   - unique on primary key
   - relationships for all foreign keys"
""")


In [ ]:
# Interview Question 5
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q5: How would you implement incremental processing?     ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"Incremental models in dbt process only new/changed data:

CONFIG:
  {{ config(
      materialized='incremental',
      unique_key='order_id',
      incremental_strategy='merge'
  ) }}

LOGIC:
  SELECT ...
  FROM {{ ref('stg_orders') }}
  {% if is_incremental() %}
  WHERE updated_at > (SELECT MAX(updated_at) FROM {{ this }})
  {% endif %}

KEY DECISIONS:

1. STRATEGY:
   - merge: UPSERT on unique_key (handles updates)
   - append: INSERT only (for immutable events)
   - delete+insert: delete matching, insert fresh

2. WATERMARK COLUMN:
   - updated_at: catches updates
   - created_at: only catches new records
   - _loaded_at: based on ingestion time

3. LATE-ARRIVING DATA:
   - Use a lookback window: WHERE date > MAX(date) - INTERVAL 3 DAYS
   - Or: full refresh periodically (dbt run --full-refresh)

4. IDEMPOTENCY:
   - merge strategy is naturally idempotent
   - append needs deduplication logic
   - Always test with: run twice → same result"
""")


In [ ]:
# Interview Question 6
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  INTERVIEW Q6: dbt vs PySpark — when to use each?                  ║
╚══════════════════════════════════════════════════════════════════════╝

STRONG ANSWER:

"Both are transformation tools, but for different use cases:

USE dbt WHEN:
  ✅ Transformations are expressible in SQL
  ✅ Team is SQL-heavy (analytics engineers)
  ✅ Data is already in the warehouse
  ✅ You need built-in testing + documentation
  ✅ Batch processing is sufficient
  ✅ You want a standardized project structure

USE PySpark WHEN:
  ✅ Complex logic that's hard in SQL (ML, NLP, graph)
  ✅ Streaming / real-time processing needed
  ✅ Heavy data engineering (file parsing, API calls)
  ✅ Need Python libraries (pandas, scikit-learn)
  ✅ Processing unstructured data (JSON, XML, images)
  ✅ Custom connectors or integrations

USE BOTH (common in practice):
  - PySpark for ingestion + complex transforms (bronze → silver)
  - dbt for analytics transforms (silver → gold)
  - PySpark handles what SQL can't
  - dbt handles what benefits from its testing/docs framework

ON DATABRICKS SPECIFICALLY:
  - dbt-databricks adapter works well
  - Lakeflow Declarative Pipelines (formerly DLT) is the native alternative
  - Many teams use dbt for the gold layer only
  - Unity Catalog provides lineage regardless of tool choice"
""")


---
# 🧹 Cleanup

In [ ]:
# Cleanup: drop views and tables created in this notebook
cleanup_views = [
    "techmart_dw.stg_orders",
    "techmart_dw.stg_customers",
    "techmart_dw.stg_products",
    "techmart_dw.stg_order_items",
    "techmart_dw.stg_payments",
    "techmart_dw.stg_shipments",
    "techmart_dw.int_order_items_enriched",
    "techmart_dw.int_customer_orders",
    "techmart_dw.int_daily_order_metrics",
    "techmart_dw.int_order_fulfillment",
    "techmart_dw.int_customer_lifetime",
    "techmart_dw.macro_demo_payments_pivot",
    "techmart_dw.macro_demo_order_keys"
]

cleanup_tables = [
    "techmart_dw.fct_daily_sales",
    "techmart_dw.dim_customers",
    "techmart_dw.fct_shipping_performance",
    "techmart_dw.fct_customer_cohorts",
    "techmart_dw.inc_daily_sales",
    "techmart_dw.inc_customer_activity",
    "techmart_dw.inc_order_summary"
]

for view in cleanup_views:
    try:
        spark.sql(f"DROP VIEW IF EXISTS {view}")
    except:
        pass

for table in cleanup_tables:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {table}")
    except:
        pass

print(f"✅ Cleaned up {len(cleanup_views)} views and {len(cleanup_tables)} tables")

## dbt Tests -- Data Quality Built In

### Generic Tests (Built-in)

```yaml
# models/staging/_stg_models.yml
version: 2

models:
  - name: stg_orders
    description: "Cleaned orders from source system"
    columns:
      - name: order_id
        description: "Unique order identifier"
        tests:
          - not_null
          - unique
      
      - name: order_status
        tests:
          - not_null
          - accepted_values:
              values: ['pending', 'completed', 'shipped', 'cancelled']
      
      - name: customer_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_customers')
              field: customer_id
      
      - name: order_total
        tests:
          - not_null
          - dbt_utils.expression_is_true:
              expression: ">= 0"
```

### Custom Tests (SQL)

```sql
-- tests/assert_positive_revenue.sql
-- This test FAILS if any rows are returned
SELECT order_id, order_total
FROM {{ ref('stg_orders') }}
WHERE order_total < 0
```

### Singular Tests (One-off assertions)

```sql
-- tests/assert_revenue_reconciliation.sql
-- Revenue in orders should match sum of order_items
SELECT
    o.order_id,
    o.order_total,
    SUM(oi.line_total) AS calculated_total
FROM {{ ref('stg_orders') }} o
JOIN {{ ref('stg_order_items') }} oi ON o.order_id = oi.order_id
GROUP BY o.order_id, o.order_total
HAVING ABS(o.order_total - SUM(oi.line_total)) > 0.01
```

### Running Tests

```bash
# Run all tests
dbt test

# Run tests for specific model
dbt test --select stg_orders

# Run only specific test type
dbt test --select test_type:generic

# Run tests and show failures
dbt test --store-failures
```

In [ ]:
# dbt test simulation
print("dbt TESTS SIMULATION")
print("=" * 60)

# Simulate running dbt tests
class DbtTestRunner:
    """Simulates dbt test execution."""
    
    def __init__(self, spark_session):
        self.spark = spark_session
        self.results = []
    
    def not_null(self, model, column):
        """Test: column should have no NULLs."""
        df = self.spark.table(f"techmart_dw.{model}")
        failures = df.filter(f"{column} IS NULL").count()
        self._record(f"{model}.{column}", "not_null", failures == 0, failures)
    
    def unique(self, model, column):
        """Test: column should have no duplicates."""
        df = self.spark.table(f"techmart_dw.{model}")
        total = df.count()
        distinct = df.select(column).distinct().count()
        failures = total - distinct
        self._record(f"{model}.{column}", "unique", failures == 0, failures)
    
    def accepted_values(self, model, column, values):
        """Test: column should only contain allowed values."""
        df = self.spark.table(f"techmart_dw.{model}")
        from pyspark.sql.functions import col
        failures = df.filter(~col(column).isin(values)).count()
        self._record(f"{model}.{column}", f"accepted_values({values})", failures == 0, failures)
    
    def positive(self, model, column):
        """Test: column should be positive."""
        df = self.spark.table(f"techmart_dw.{model}")
        from pyspark.sql.functions import col
        failures = df.filter(col(column) <= 0).count()
        self._record(f"{model}.{column}", "positive", failures == 0, failures)
    
    def _record(self, target, test_name, passed, failures):
        self.results.append({
            "target": target,
            "test": test_name,
            "passed": passed,
            "failures": failures
        })
    
    def report(self):
        passed = sum(1 for r in self.results if r["passed"])
        failed = sum(1 for r in self.results if not r["passed"])
        
        print(f"\n{'='*60}")
        print(f"dbt test results: {passed} passed, {failed} failed")
        print(f"{'='*60}")
        for r in self.results:
            status = "PASS" if r["passed"] else "FAIL"
            icon = "✅" if r["passed"] else "❌"
            failures_str = f" ({r['failures']} failures)" if not r["passed"] else ""
            print(f"  {icon} [{status}] {r['target']} -- {r['test']}{failures_str}")
        
        if failed > 0:
            print(f"\n❌ {failed} test(s) failed!")
        else:
            print(f"\n✅ All {passed} tests passed!")


# Run tests
runner = DbtTestRunner(spark)

# Test stg_orders
runner.not_null("stg_orders", "order_id")
runner.unique("stg_orders", "order_id")
runner.not_null("stg_orders", "customer_id")
runner.accepted_values("stg_orders", "order_status",
                       ["pending", "completed", "shipped", "cancelled"])
runner.positive("stg_orders", "order_total")

# Test stg_customers
runner.not_null("stg_customers", "customer_id")
runner.unique("stg_customers", "customer_id")
runner.not_null("stg_customers", "email")

# Test fct_daily_sales
runner.not_null("fct_daily_sales", "sale_date")
runner.positive("fct_daily_sales", "total_revenue")

runner.report()


In [ ]:
# ============================================
# 🎯 YOUR TURN: Build a dbt-Style Pipeline
# ============================================
# Using the techmart_dw database, build a dbt-style pipeline:
#
# 1. Create stg_payments (view):
#    - payment_id, order_id, LOWER(payment_method) as payment_method,
#    - CAST(amount AS DECIMAL(12,2)) as payment_amount,
#    - LOWER(payment_status) as payment_status,
#    - CAST(payment_date AS DATE) as paid_at
#    - Filter: payment_id IS NOT NULL
#
# 2. Create int_order_payments (view):
#    - Join stg_orders + stg_payments on order_id
#    - Include: order_id, customer_id, order_total, order_status,
#               payment_method, payment_amount, payment_status
#
# 3. Create fct_payment_summary (table):
#    - Group by payment_method
#    - Columns: payment_method, total_transactions, total_amount,
#               avg_amount, success_rate (% where payment_status='completed')
#
# Write your SQL below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import col, count, sum as spark_sum, avg, round as spark_round, when

# 1. Staging
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.stg_payments AS
SELECT
    payment_id,
    order_id,
    LOWER(TRIM(payment_method)) AS payment_method,
    CAST(amount AS DECIMAL(12,2)) AS payment_amount,
    LOWER(TRIM(payment_status)) AS payment_status,
    CAST(payment_date AS DATE) AS paid_at
FROM techmart_dw.payments
WHERE payment_id IS NOT NULL
""")
print(f"stg_payments: {spark.table('techmart_dw.stg_payments').count():,} rows")

# 2. Intermediate
spark.sql("""
CREATE OR REPLACE VIEW techmart_dw.int_order_payments AS
SELECT
    o.order_id,
    o.customer_id,
    o.order_total,
    o.order_status,
    p.payment_method,
    p.payment_amount,
    p.payment_status,
    p.paid_at
FROM techmart_dw.stg_orders o
LEFT JOIN techmart_dw.stg_payments p ON o.order_id = p.order_id
""")
print(f"int_order_payments: {spark.table('techmart_dw.int_order_payments').count():,} rows")

# 3. Mart
spark.sql("DROP TABLE IF EXISTS techmart_dw.fct_payment_summary")
spark.sql("""
CREATE TABLE techmart_dw.fct_payment_summary AS
SELECT
    payment_method,
    COUNT(*) AS total_transactions,
    ROUND(SUM(payment_amount), 2) AS total_amount,
    ROUND(AVG(payment_amount), 2) AS avg_amount,
    ROUND(
        SUM(CASE WHEN payment_status = 'completed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS success_rate_pct
FROM techmart_dw.stg_payments
GROUP BY payment_method
ORDER BY total_amount DESC
""")
print(f"\nfct_payment_summary:")
spark.table("techmart_dw.fct_payment_summary").show()


---
# 📋 Summary

## What You Learned

| Topic | Key Takeaway |
|---|---|
| **What is dbt** | SQL-first transformation framework (the T in ELT) |
| **Project Structure** | models/ (staging→intermediate→marts), tests/, macros/ |
| **Models** | SELECT statements that become tables/views automatically |
| **Materializations** | view, table, incremental, ephemeral |
| **ref()** | Dependency management between models |
| **Testing** | not_null, unique, accepted_values, relationships |
| **Macros** | Reusable SQL templates (Jinja → Python f-strings) |
| **Incremental** | Process only new data using MERGE + watermarks |
| **Documentation** | Column descriptions + auto-generated catalog |
| **dbt on Databricks** | dbt-databricks adapter + Unity Catalog integration |

## dbt Patterns You Can Apply Anywhere

Even without dbt installed, these patterns improve any data project:
1. **Layered models** — staging → intermediate → marts
2. **One model per file** — modular, testable, reusable
3. **Tests as code** — automated quality checks
4. **Documentation with code** — never stale
5. **Incremental processing** — efficient at scale

## Next Steps
- **Notebook 30:** Capstone — Streaming Platform
- **Notebook 35:** Declarative Automation Bundles (DABs) deep dive

---
*Notebook 29 of the Databricks Data Engineering series*
*Study Order: 20*